# 05 · Modo de operación local del moderador

Este cuaderno publica una página HTML autocontenida en un servidor local. La interfaz usa una sola caja y detecta automáticamente si la entrada es una frase o un enlace de YouTube. El backend ejecuta el mejor modelo clásico, el mejor MiniLM Transformer, el Qwen fine-tuned operativo, la comparación de los tres o su consenso.

Los videos sin subtítulos manuales ni automáticos se rechazan. Los subtítulos se agrupan con ventanas de 30 segundos/600 caracteres, como en preentrenamiento, y cada alerta conserva un enlace temporal. Inferencias y revisiones humanas se guardan localmente por modelo y categoría; las revisiones se exportan a JSONL para reentrenamiento futuro.

La aplicación es de apoyo humano: no autoriza bloqueo, sanción ni moderación autónoma.

In [ ]:
from importlib.util import find_spec
import subprocess, sys

DEPENDENCIAS = {
    'yt_dlp': 'yt-dlp>=2025.6',
    'transformers': 'transformers>=4.51,<6',
    'peft': 'peft>=0.15,<1',
    'huggingface_hub': 'huggingface-hub>=0.30',
    'sklearn': 'scikit-learn>=1.4',
}
faltantes = [paquete for modulo, paquete in DEPENDENCIAS.items() if find_spec(modulo) is None]
if faltantes:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *faltantes], check=True)
print('Dependencias listas.')

## 2. Crear `05_frontend_despliegue`

Esta etapa crea una carpeta portable con HTML, backend, base SQLite nueva, registro y hashes, los tres modelos seleccionados, Qwen base completo y archivos Docker. Requiere la ejecución final en orden `04_207 → 04_208`; se detiene si la auditoría no corresponde a la selección Qwen actual o conserva pendientes. No copia estadísticas ni revisiones humanas existentes. El consenso incluido es mayoritario: **2 de 3**, no unanimidad.

In [ ]:
# Parámetros de empaquetado. El resultado ocupa aproximadamente 1.7 GiB.
CREAR_CARPETA_DESPLIEGUE = True
RECREAR_CARPETA_DESPLIEGUE = True
DESCARGAR_QWEN_BASE_SI_FALTA = True

if CREAR_CARPETA_DESPLIEGUE:
    from scripts_auxiliares import crear_bundle_despliegue_05 as bundle05
    BUNDLE_05 = bundle05.build_deployment_bundle(
        REGISTRY,
        recreate=RECREAR_CARPETA_DESPLIEGUE,
        download_qwen_base_if_needed=DESCARGAR_QWEN_BASE_SI_FALTA,
    )
    display({
        'carpeta': BUNDLE_05['output_dir'],
        'tamaño_GiB': round(BUNDLE_05['total_gib'], 3),
        'archivos': len(BUNDLE_05['files']),
        'consenso': BUNDLE_05['consensus'],
        'modelos': BUNDLE_05['models'],
    })
else:
    print('Creación del bundle omitida por configuración.')

## 1. Verificar artefactos y construir el registro desplegable

Si el checkpoint E5 o los artefactos Qwen sólo están en Drive, se recuperan sin sobrescribir conflictos locales. La selección se rehace únicamente con métricas de validation; test no interviene.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys
import pandas as pd
from IPython.display import display, Markdown

ROOT = Path.cwd().resolve()
if ROOT.name.lower() == 'cuadernos': ROOT = ROOT.parent
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

DRIVE_BUNDLE = Path(r'G:\My Drive\PLN_colab_04_artifacts')
RECOVERY_SCRIPT = ROOT / 'scripts_auxiliares' / 'recuperar_resultados_colab_04_20x.ps1'
DEPLOYMENT_REQUIRED = [
    ROOT / 'modelos/transformer_plano_4/e5_small/best_checkpoint.pt',
    ROOT / 'modelos/transformer_plano_4/e5_small/tokenizer/tokenizer.json',
    ROOT / 'resultados/metricas/qwen3_06b_lora_acoso_amenaza_4/seleccion_operativa_validacion.json',
    ROOT / 'modelos/qwen3_06b_lora_acoso_amenaza_4/epoch_adapters/epoch_03/adapter_model.safetensors',
]
missing_deployment = [path for path in DEPLOYMENT_REQUIRED if not path.is_file()]
if missing_deployment and sys.platform == 'win32' and DRIVE_BUNDLE.is_dir() and RECOVERY_SCRIPT.is_file():
    recovery = subprocess.run([
        'powershell', '-NoProfile', '-ExecutionPolicy', 'Bypass', '-File', str(RECOVERY_SCRIPT),
        '-Workspace', str(ROOT), '-DriveBundle', str(DRIVE_BUNDLE), '-DeploymentOnly',
    ], text=True, capture_output=True)
    if recovery.returncode != 0: raise RuntimeError(recovery.stderr or recovery.stdout)
    print(recovery.stdout.strip())
missing_deployment = [str(path.relative_to(ROOT)) for path in DEPLOYMENT_REQUIRED if not path.is_file()]
if missing_deployment: raise FileNotFoundError('Faltan artefactos de operación:\n' + '\n'.join(missing_deployment))

from scripts_auxiliares import registro_modelos_produccion_4 as registry4
from scripts_auxiliares import servidor_moderacion_05 as moderation05
REGISTRY = registry4.build_registry()
SERVICE = moderation05.ModerationService(REGISTRY)
display(pd.DataFrame([
    {'tipo': slot, 'modelo': model['label'], 'PR-AUC validation': model['validation_damage_pr_auc_macro'], 'test_usado_para_seleccion': model['test_used_for_selection']}
    for slot, model in REGISTRY['models'].items()
]))

## 3. Ejecutar una entrada directamente en el cuaderno

La celda siguiente contiene todas las variables de operación. `TIPO_ENTRADA='auto'` reconoce YouTube o texto igual que la página. Cambie `ENTRADA` y ejecútela; no es necesario iniciar el servidor.

In [ ]:
# Parámetros de ejecución directa (todos están deliberadamente en esta celda).
ENTRADA = ''  # frase o enlace de YouTube
TIPO_ENTRADA = 'auto'  # auto | text | youtube
MODO_MODELO = 'consensus'  # classical | transformer | qwen | compare | consensus
IDIOMAS_SUBTITULOS = ('es', 'es-419', 'es-US', 'en')
MAX_CHUNKS = 300
GUARDAR_ESTADISTICAS = True

if ENTRADA.strip():
    RESULTADO = SERVICE.analyze(
        ENTRADA,
        mode=MODO_MODELO,
        input_type=TIPO_ENTRADA,
        subtitle_languages=IDIOMAS_SUBTITULOS,
        max_chunks=MAX_CHUNKS,
        persist=GUARDAR_ESTADISTICAS,
    )
    display({key: RESULTADO[key] for key in ('analysis_id', 'input_type', 'mode', 'summary')})
    display(pd.DataFrame([
        {
            'chunk_id': chunk['chunk_id'], 'inicio': chunk['start_seconds'],
            'modelo': result['model_label'], 'etiquetas': result['predicted_labels'],
            'confianza': result['confidence'], 'requiere_revision': result['requires_review'],
            'enlace': chunk['watch_url'],
        }
        for chunk in RESULTADO['chunks'] for result in chunk['results']
    ]))
else:
    print('Defina ENTRADA para ejecutar inferencia directa.')

## 4. Iniciar la página HTML local

El servidor escucha sólo en `127.0.0.1` por defecto. El HTML, CSS, JavaScript y ayuda están en un único archivo; la inferencia se realiza en el backend local con los checkpoints verificados.

In [ ]:
HOST = '127.0.0.1'
PORT = 8765  # use 0 para elegir un puerto libre
ABRIR_NAVEGADOR = True
PERMITIR_RED = False

if 'SERVER_05' in globals():
    SERVER_05.stop()
SERVER_05 = moderation05.start_server(
    SERVICE, host=HOST, port=PORT, open_browser=ABRIR_NAVEGADOR, allow_network=PERMITIR_RED
)
display(Markdown(f'Aplicación disponible en **[{SERVER_05.url}]({SERVER_05.url})**'))
print('HTML:', moderation05.HTML_PATH.relative_to(ROOT))
print('SQLite:', moderation05.DATABASE_PATH.relative_to(ROOT))
print('Revisiones para reentrenamiento:', moderation05.RETRAINING_JSONL_PATH.relative_to(ROOT))

## 5. Detener el servidor

Ejecute esta celda antes de cerrar el kernel si quiere liberar el puerto inmediatamente.

In [ ]:
if 'SERVER_05' in globals():
    SERVER_05.stop()
    del SERVER_05
    print('Servidor 05 detenido.')
else:
    print('No hay servidor 05 activo.')

## Documentación

La guía completa de modos, revisión, estadísticas, reentrenamiento y límites está en `Cuadernos/05_MODO_OPERACION.md`. La misma ayuda resumida está disponible dentro de la página.